In [1]:
from __future__ import print_function, division
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.datasets import MNIST, CIFAR10, CIFAR100, SVHN

import os
import torch
from torch import nn,optim
import torch.nn.functional as F
from torch.autograd import Variable
from torchvision import datasets, transforms

from time import perf_counter

import  numpy as np
import torch.utils.data as Data

from torch.utils.data import Dataset, DataLoader



import os, random, time, copy
import numpy as np
import pandas as pd
import os.path as path
import scipy.io as sio
from scipy import misc
from scipy import ndimage, signal
import scipy
import pickle
import sys
import math
import matplotlib.pyplot as plt
import PIL.Image
from io import BytesIO


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler 
import torch.nn.functional as F
from torch.autograd import Variable
import torchvision
from torchvision import datasets, models, transforms
import torchvision.utils as vutils


import warnings 
warnings.filterwarnings("ignore")
print(sys.version)
print(torch.__version__)




class Safeman(Dataset):
    
    def __init__(self, data,targets):
        super(Safeman, self).__init__()
        self.data = data
        self.targets = targets
        
     
    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img, target = self.data[idx], self.targets[idx]
        return img, target

class Safeman_Filter(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)
        
class Safeman_FilterB(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        new_targets = []
        for i in range(len(targets)):
            if targets[i] in known:
                new_targets.append(0)
            else:
                new_targets.append(1)
        self.targets = np.array(new_targets)
        self.data = self.data

class Safeman_FilterC(Safeman):
    
    def __Filter__(self, trainknown):
        train_class_num=len(trainknown)
        for i in range(0,len(self.targets)) :
            if self.targets[i]>train_class_num:
                self.targets[i] = train_class_num
        self.data = self.data

        
        
class Safeman_FilterF(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                dd = known.index(targets[i])
                if dd == 4:
                    new_targets.append(0)
                else:
                    new_targets.append(1)                   
                    
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)   


        
def setup_seed(seed):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(8)



known=[0, 1, 2,3,4,5,6,7]


X_train0 = np.load('./TONdataset/x_train_iot1028+1del.npy')
y_train1 = np.load('./TONdataset/y_train_iot1028+1del.npy')
X_final_test0 = np.load('./TONdataset/x_test_iot1028+1del.npy' )
y__final_test1 = np.load('./TONdataset/y_test_iot1028+1del.npy')


X_train1=[]
X_final_test1=[]

for i in range(len(y_train1)):
    a = np.resize(X_train0[i], (21))
    X_train1 += [a]
    
for j in range(len(y__final_test1)):
    b = np.resize(X_final_test0[j], (21))
    X_final_test1 += [b]

i=0
j=0



x_train, x_test, y_train,y_test = torch.Tensor(X_train1), torch.Tensor(X_final_test1), torch.from_numpy(y_train1), torch.from_numpy(y__final_test1)

print(x_train.shape, x_test.shape, y_train.shape,y_test.shape)


train_dataset = Data.TensorDataset(x_train, y_train)
train_dataset.data = train_dataset.tensors[0]
train_dataset.targets = train_dataset.tensors[1]



test_dataset = Data.TensorDataset(x_test, y_test)
test_dataset.data = test_dataset.tensors[0]
test_dataset.targets = test_dataset.tensors[1]

labels =['backdoor', 'ddos', 'dos', 'injection', 'normal', 'password', 'scanning', 'xss']

train_dataset.classes = labels
test_dataset.classes = labels

train_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}
test_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}

num_class=len(labels)

b_s=256

trainset = Safeman_Filter(data=train_dataset.data,targets=train_dataset.targets)
print('All down Train Data:', len(trainset))
trainset.__Filter__(known=known)


train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real train Data:', len(trainset))



testsetA = Safeman_FilterF(data=test_dataset.data,targets=test_dataset.targets)
print('All testsetA Data:', len(testsetA))
testsetA.__Filter__(known=known)


test_loader_A = torch.utils.data.DataLoader(
    testsetA, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real testsetA Data:', len(testsetA))


print("done!")

3.7.16 (default, Jan 17 2023, 22:20:44) 
[GCC 11.2.0]
1.13.1
torch.Size([242000, 21]) torch.Size([48000, 21]) torch.Size([242000]) torch.Size([48000])
All down Train Data: 242000
Real train Data: 242000
All testsetA Data: 48000
Real testsetA Data: 48000
done!


In [2]:
unique,counts = np.unique(trainset.targets,return_counts=True)
print(unique, counts)

[0 1 2 3 4 5 6 7] [ 16000  16000  16000  16000 130000  16000  16000  16000]


In [3]:
X_trainset_data=trainset.data
X_trainset_targets=trainset.targets

In [4]:
unique,counts = np.unique(X_trainset_targets,return_counts=True)
print(unique, counts)

[0 1 2 3 4 5 6 7] [ 16000  16000  16000  16000 130000  16000  16000  16000]


In [5]:
X_trainset_targets

array([6, 6, 6, ..., 0, 0, 0])

In [6]:
count=[10, 10, 10, 10, 130000, 10, 10, 10]
num_class=8
lists = [[] for i in range(num_class)]
y_train_temp=[]
x_train_temp=[]

In [7]:
for i in range(len(X_trainset_targets)):
    if X_trainset_targets[i]==4:
        pass
    else:
        if len(lists[X_trainset_targets[i]])<count[X_trainset_targets[i]]:
            lists[X_trainset_targets[i]].append(X_trainset_targets[i])   
            y_train_temp+=[X_trainset_targets[i]]
            a = np.resize(X_trainset_data[i], (3, 32, 32))
            x_train_temp += [a]

In [8]:
unique,counts = np.unique(y_train_temp,return_counts=True)
print(unique, counts)

[0 1 2 3 5 6 7] [10 10 10 10 10 10 10]


In [9]:
x_train2, y_train2 = torch.Tensor(x_train_temp), torch.Tensor(y_train_temp)

print(x_train2.shape, y_train2.shape)

train_dataset2 = Data.TensorDataset(x_train2, y_train2)
train_dataset2.data = train_dataset2.tensors[0]
train_dataset2.targets = train_dataset2.tensors[1]


trainset2 = Safeman_FilterF(data=train_dataset2.data,targets=train_dataset2.targets)
print('All down Train Data:', len(trainset2))
trainset2.__Filter__(known=known)

b_s=1

train_loader2 = torch.utils.data.DataLoader(
    trainset2, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real train Data2:', len(trainset2))

torch.Size([70, 3, 32, 32]) torch.Size([70])
All down Train Data: 70
Real train Data2: 70


In [10]:
manualSeed = 999
print("Random Seed: ", manualSeed)
random.seed(manualSeed)
torch.manual_seed(manualSeed)
torch.manual_seed(0)
device ='cpu'
if torch.cuda.is_available(): 
    device='cuda:0'


total_epoch_num = 5 

lr = 0.0001 

num_epochs = total_epoch_num
torch.cuda.device_count()
torch.cuda.empty_cache()

save_dir="./res1030/dcgan2nslmal"

print(save_dir)    
if not os.path.exists(save_dir): os.makedirs(save_dir)


Random Seed:  999
./res1030/dcgan2nslmal


In [11]:
      
class Generatorzy(nn.Module):
    def __init__(self, z_dim):
        super(Generatorzy, self).__init__()
        
        self.fc = nn.Linear(z_dim, 256*8*8)
        self.g_deconv_1 = nn.Sequential(
                          nn.ConvTranspose2d(256, 128, kernel_size=3,
                                    stride= 2, padding=(3-2+1)//2,
                                    output_padding = (3-2)%2), 
                          nn.BatchNorm2d(128),
                          nn.LeakyReLU()
                          )
        self.g_deconv_2 = nn.Sequential(
                          nn.ConvTranspose2d(128, 64, kernel_size=3,
                                    stride= 1, padding=(3-1+1)//2,
                                    output_padding = (3-1)%2), 
                          nn.BatchNorm2d(64),
                          nn.LeakyReLU()
                          )
        self.g_deconv_3 = nn.Sequential(
                          nn.ConvTranspose2d(64, 3, kernel_size=3,
                                    stride= 2, padding=(3-2+1)//2,
                                    output_padding = (3-2)%2),
                          nn.Tanh()
                          )
        self.fczy = nn.Linear(3*32*32, 21)  
        

    def forward(self, x):
        x = self.fc(x).view(-1, 256, 8, 8)
        x = self.g_deconv_1(x)
        x = self.g_deconv_2(x)
        x = self.g_deconv_3(x)
        x_zy = self.fczy(x.view(-1,3*32*32))
        return x_zy,x     


    
class Discriminatorzy(nn.Module):
    def __init__(self):
        super(Discriminatorzy, self).__init__()
        
        self.d_conv_1 = nn.Sequential(
                          nn.Conv2d(3, 32, kernel_size=3,
                                    stride=2, padding=1), 
                          nn.LeakyReLU()
                          )
        self.d_conv_2 = nn.Sequential(
                          nn.Conv2d(32, 64, kernel_size=3,
                                    stride=2, padding=1), 
                          nn.LeakyReLU()
                          )
        self.d_conv_3 = nn.Sequential(
                          nn.Conv2d(64, 128, kernel_size=3,
                                    stride=2, padding=0), 
                          nn.LeakyReLU()
                          )
        self.fc = nn.Linear(3*3*128, 1)
        self.fczy = nn.Linear(3*3*128, 46)  

    def forward(self, x):
        x = self.d_conv_1(x)
        x = self.d_conv_2(x)
        x = self.d_conv_3(x)
        x = x.view(-1, 128*3*3)
        x_zy = self.fczy(x)
        x = torch.sigmoid(self.fc(x))
        return x_zy,x

In [12]:
print(device)

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)     
        


z_dim=100
batch_size=b_s
discriminator = Discriminatorzy()
netDzy = discriminator.to(device)


generator = Generatorzy(z_dim)
netGzy = generator.to(device)


netDzy.apply(weights_init)
netGzy.apply(weights_init)

criterion = nn.BCELoss()

real_label = 1
fake_label = 0
beta1 = 0.5

optimizerD = optim.Adam(netDzy.parameters(), lr=lr/1.5, betas=(beta1, 0.999))

optimizerG = optim.Adam(netGzy.parameters(), lr=lr, betas=(beta1, 0.999))


cpu


In [13]:

dataloader_train_closeset=train_loader2
print("Starting Training Loop...")
# For each epoch
for epoch in range(num_epochs):
    i=0
    for sample in dataloader_train_closeset:
        data, datalabel = sample
        ############################
        # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
        ###########################

        netDzy.zero_grad()

        real_cpu = data.to(device)

        bb_size = real_cpu.size(0)

        label = torch.full((bb_size,), real_label, device=device)

        _,output = netDzy(real_cpu)
        output=output.view(-1)

        output=output.to(torch.float32)
        label=label.to(torch.float32)
        errD_real = criterion(output, label)

        errD_real.backward()
        

        
        D_x = output.mean().item()


        noise = torch.randn(batch_size, z_dim, device=device)
        _,fake = netGzy(noise)
        label.fill_(fake_label)        


        _,output = netDzy(fake.detach())
        
        output=output.view(-1)

        output=output.to(torch.float32)
        
        
        label=label.to(torch.float32)

        
        errD_fake = criterion(output, label)
        errD_fake.backward()

        D_G_z1 = output.mean().item()

        errD = errD_real + errD_fake

        optimizerD.step()

        

        ############################
        # (2) Update G network: maximize log(D(G(z)))
        ###########################
        netGzy.zero_grad()
        label.fill_(real_label)  

        _,output = netDzy(fake)
        output=output.view(-1)

        output=output.to(torch.float32)
        label=label.to(torch.float32)
        errG = criterion(output, label)
        errG.backward()
        
        
        D_G_z2 = output.mean().item()
        # Update G
        optimizerG.step()

        # Output training stats
        if i % 20 == 0:
            print('[%d/%d][%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f'
                  % (epoch, num_epochs, i, len(dataloader_train_closeset),
                     errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

        i+=1
        
        
    cur_model_wts = copy.deepcopy(netGzy.state_dict())
    path_to_save_paramOnly = os.path.join(save_dir, 'dcgan-2nsl-epoch-{}.GNet'.format(epoch+1))
    torch.save(cur_model_wts, path_to_save_paramOnly)
    
    cur_model_wts = copy.deepcopy(netDzy.state_dict())
    path_to_save_paramOnly = os.path.join(save_dir, 'dcgan-2nsl-epoch-{}.DNet'.format(epoch+1))
    torch.save(cur_model_wts, path_to_save_paramOnly)


Starting Training Loop...
[0/5][0/70]	Loss_D: 1.3869	Loss_G: 0.6858	D(x): 0.5034	D(G(z)): 0.5037 / 0.5037
[0/5][20/70]	Loss_D: 1.3764	Loss_G: 0.6831	D(x): 0.5103	D(G(z)): 0.5052 / 0.5051
[0/5][40/70]	Loss_D: 1.3747	Loss_G: 0.6926	D(x): 0.5069	D(G(z)): 0.5010 / 0.5003
[0/5][60/70]	Loss_D: 1.3441	Loss_G: 0.7044	D(x): 0.5162	D(G(z)): 0.4948 / 0.4944
[1/5][0/70]	Loss_D: 1.3413	Loss_G: 0.7102	D(x): 0.5146	D(G(z)): 0.4918 / 0.4915
[1/5][20/70]	Loss_D: 1.2592	Loss_G: 0.7223	D(x): 0.5541	D(G(z)): 0.4877 / 0.4856
[1/5][40/70]	Loss_D: 1.1523	Loss_G: 0.8515	D(x): 0.5514	D(G(z)): 0.4271 / 0.4268
[1/5][60/70]	Loss_D: 1.5320	Loss_G: 0.5384	D(x): 0.5425	D(G(z)): 0.6016 / 0.5837
[2/5][0/70]	Loss_D: 1.0441	Loss_G: 0.9212	D(x): 0.5858	D(G(z)): 0.3991 / 0.3981
[2/5][20/70]	Loss_D: 1.2202	Loss_G: 0.6940	D(x): 0.6035	D(G(z)): 0.5109 / 0.4996
[2/5][40/70]	Loss_D: 1.1939	Loss_G: 0.8766	D(x): 0.5191	D(G(z)): 0.4162 / 0.4162
[2/5][60/70]	Loss_D: 1.3655	Loss_G: 0.7714	D(x): 0.4732	D(G(z)): 0.4605 / 0.4624
[3/5]